# Étape 2 — Test décisif sur les données (MOOCCubeX + MoocRadar)

Ce notebook répond à une seule question : **a-t-on assez de prérequis annotés, reliés à des exercices et à des réponses d'apprenants, pour mener l'étude ?**

Exécutez les cellules dans l'ordre. Les résultats sont enregistrés dans `projet_kt/resultats/`.
À la fin, **copiez le résumé de la dernière cellule** (et le fichier `resultats/test_decisif.json`) dans la conversation.

Durée : quelques minutes, sauf la cellule 6 (optionnelle, 30 à 90 min) qui parcourt les 21 Go de réponses de MOOCCubeX.

In [ ]:
# Cellule 1 — Connexion à Google Drive et chemins
from google.colab import drive
drive.mount('/content/drive')

import os
DOSSIER   = '/content/drive/MyDrive/projet_kt'          # à adapter si besoin
ZIP_X     = os.path.join(DOSSIER, 'data_moccub.zip')    # MOOCCubeX
ZIP_R     = os.path.join(DOSSIER, 'MOOCRadar.zip')      # MoocRadar
PREREQ    = os.path.join(DOSSIER, 'prerequisites')      # fichiers de prérequis (téléchargés en cellule 2)
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(PREREQ, exist_ok=True)
os.makedirs(RESULTATS, exist_ok=True)

for p in [ZIP_X, ZIP_R]:
    print(('OK      ' if os.path.exists(p) else 'MANQUANT'), p)

**Important :** votre `data_moccub.zip` est bien MOOCCubeX, mais il **ne contient pas les fichiers de prérequis**
(`psy.json`, `math.json`, `cs.json`). Ce sont pourtant le cœur de l'étude. La cellule 2 les télécharge (environ 280 Mo) dans votre Drive.

In [ ]:
# Cellule 2 — Téléchargement des fichiers de prérequis de MOOCCubeX (une seule fois)
BASE = 'https://lfs.aminer.cn/misc/moocdata/data/mooccube2/prerequisites'
DOMAINES = {'cs': 'cs.json', 'math': 'math.json', 'psy': 'psy.json'}
for dom, nom in DOMAINES.items():
    cible = os.path.join(PREREQ, nom)
    if os.path.exists(cible) and os.path.getsize(cible) > 1_000_000:
        print('déjà présent :', cible)
        continue
    !wget -q -c -O "{cible}" "{BASE}/{nom}"
    print(nom, '->', round(os.path.getsize(cible) / 1e6, 1), 'Mo')

In [ ]:
# Cellule 3 — Inspection du format des fichiers de prérequis
import json, itertools
from collections import Counter

def lire_enregistrements(chemin, limite=None):
    # Gère JSON Lines, tableau JSON, dictionnaire JSON et texte tabulé.
    with open(chemin, 'r', encoding='utf-8') as f:
        debut = f.read(1)
        f.seek(0)
        if debut == '[':
            donnees = json.load(f)
            iterable = iter(donnees)
        elif debut == '{':
            premiere = f.readline()
            try:
                json.loads(premiere)          # JSON Lines
                f.seek(0)
                iterable = (json.loads(l) for l in f if l.strip())
            except json.JSONDecodeError:      # un seul gros dictionnaire
                f.seek(0)
                donnees = json.load(f)
                iterable = ({'_cle': k, '_valeur': v} for k, v in donnees.items())
        else:                                 # texte tabulé
            iterable = (l.rstrip('\n').split('\t') for l in f if l.strip())
        for i, rec in enumerate(iterable):
            if limite is not None and i >= limite:
                break
            yield rec

for dom, nom in DOMAINES.items():
    chemin = os.path.join(PREREQ, nom)
    print('=' * 80, '\n', nom)
    recs = list(lire_enregistrements(chemin, limite=3))
    for r in recs:
        print(json.dumps(r, ensure_ascii=False)[:600])
    # Distribution des petits champs : c'est là que se trouve en général le drapeau « annoté / prédit »
    if recs and isinstance(recs[0], dict):
        compteurs = {}
        for r in lire_enregistrements(chemin, limite=200_000):
            for k, v in r.items():
                if isinstance(v, (str, int, float, bool)) or v is None:
                    compteurs.setdefault(k, Counter())[v] += 1
        for k, c in compteurs.items():
            if len(c) <= 20:
                print(f'  champ « {k} » : {dict(c.most_common(20))}')
            else:
                print(f'  champ « {k} » : {len(c)} valeurs distinctes (ex. {list(c)[:3]})')

In [ ]:
# Cellule 4 — Statistiques de MoocRadar (lecture directe dans le zip)
import zipfile, statistics
import pandas as pd

with zipfile.ZipFile(ZIP_R) as z:
    noms = {os.path.basename(n): n for n in z.namelist()}
    with z.open(noms['problem.json']) as f:
        problemes_r = [json.loads(l) for l in f if l.strip()]
    with z.open(noms['student-problem-fine.json']) as f:
        sequences_r = json.load(f)

concepts_par_pb = {p['problem_id']: p.get('concepts') or [] for p in problemes_r}
tous_concepts_r = Counter(c for cs in concepts_par_pb.values() for c in cs)

interactions = [it for s in sequences_r for it in s['seq']]
inter_par_pb = Counter(it['problem_id'] for it in interactions)
longueurs = [len(s['seq']) for s in sequences_r]

stats_r = {
    'problemes': len(problemes_r),
    'problemes_avec_concepts': sum(1 for c in concepts_par_pb.values() if c),
    'concepts_distincts': len(tous_concepts_r),
    'cours': len({p['course_id'] for p in problemes_r}),
    'apprenants': len({it['user_id'] for it in interactions}),
    'interactions': len(interactions),
    'taux_reussite': round(sum(it['is_correct'] for it in interactions) / max(1, len(interactions)), 3),
    'longueur_sequence_mediane': statistics.median(longueurs) if longueurs else 0,
    'problemes_avec_reponses': sum(1 for p in concepts_par_pb if inter_par_pb[p] > 0),
    'knowledge_type': dict(Counter(p.get('knowledge_type') for p in problemes_r)),
    'cognitive_dimension': dict(Counter(p.get('cognitive_dimension') for p in problemes_r)),
}
for k, v in stats_r.items():
    print(f'{k:30s} {v}')

In [ ]:
# Cellule 5 (v2) — Le test décisif, lecture indépendante du format des prérequis
import zipfile, json
from collections import Counter

def id_vers_nom(c):
    # 'K_<nom>_<discipline>' -> '<nom>' ; un nom simple reste tel quel
    c = str(c).strip()
    if c.startswith('K_'):
        nom, _, _ = c[2:].rpartition('_')
        return nom or c[2:]
    return c

# --- 0. Diagnostic : les fichiers sont-ils bien téléchargés ? ---
for dom, nom in DOMAINES.items():
    chemin = os.path.join(PREREQ, nom)
    taille = os.path.getsize(chemin) if os.path.exists(chemin) else 0
    with open(chemin, 'rb') as f:
        debut = f.read(300)
    print(f'{nom}: {taille/1e6:.1f} Mo | début : {debut[:200]!r}')
    if taille < 100_000 or debut.lstrip()[:1] == b'<':
        print('   ⚠️ Fichier vide ou page HTML : relancez la cellule 2 après avoir supprimé ce fichier.')

# --- 1. Lecture : détection automatique des 2 colonnes « concept » et des colonnes « drapeau » ---
def colonnes(rec):
    if isinstance(rec, dict):
        if '_cle' in rec:
            return None
        return {k: v for k, v in rec.items() if isinstance(v, (str, int, float, bool)) or v is None}
    if isinstance(rec, (list, tuple)):
        return {f'col{i}': v for i, v in enumerate(rec)}
    return None

paires = []   # (domaine, A, B, attributs) ; A est prérequis de B dans l'ordre du fichier
for dom, nom in DOMAINES.items():
    chemin = os.path.join(PREREQ, nom)
    echantillon = [c for c in (colonnes(r) for r in lire_enregistrements(chemin, limite=5000)) if c]
    n_avant = len(paires)
    if echantillon:
        valeurs = {}
        for c in echantillon:
            for k, v in c.items():
                valeurs.setdefault(k, set()).add(v)
        ordre = list(echantillon[0].keys())
        cles_concepts = [k for k in ordre
                         if all(isinstance(v, str) for v in valeurs[k]) and len(valeurs[k]) > 20][:2]
        cles_drapeaux = [k for k in ordre if k not in cles_concepts and len(valeurs[k]) <= 10]
        print(f'{nom}: colonnes concepts = {cles_concepts} | colonnes drapeaux = {cles_drapeaux}')
        if len(cles_concepts) == 2:
            ka, kb = cles_concepts
            for r in lire_enregistrements(chemin):
                c = colonnes(r)
                if c and c.get(ka) and c.get(kb):
                    paires.append((dom, id_vers_nom(c[ka]), id_vers_nom(c[kb]),
                                   {k: c.get(k) for k in cles_drapeaux}))
    else:   # dictionnaire concept -> liste de prérequis
        for r in lire_enregistrements(chemin):
            if not isinstance(r, dict):
                continue
            items = [(r['_cle'], r['_valeur'])] if '_cle' in r else list(r.items())
            for cible, valeur in items:
                vals = valeur if isinstance(valeur, list) else [valeur]
                for v in vals:
                    src = v if isinstance(v, str) else (v.get('id') or v.get('name')) if isinstance(v, dict) else None
                    if src:
                        paires.append((dom, id_vers_nom(src), id_vers_nom(cible), {}))
    print(f'   -> {len(paires) - n_avant} paires lues')

if not paires:
    raise SystemExit("Toujours 0 paire : copiez dans la conversation la sortie de la cellule 3 ET le diagnostic ci-dessus.")

drapeaux = {}
for *_, at in paires:
    for k, v in at.items():
        drapeaux.setdefault(k, Counter())[v] += 1
print('Valeurs des drapeaux :', {k: dict(c) for k, c in drapeaux.items()})

# --- 2. MOOCCubeX : nom de concept -> problèmes ---
concept_pbs = {}
with zipfile.ZipFile(ZIP_X) as z:
    nom_cp = next(n for n in z.namelist() if n.endswith('concept-problem.txt'))
    with z.open(nom_cp) as f:
        for l in f:
            c, _, p = l.decode('utf-8').rstrip('\n').partition('\t')
            if p:
                concept_pbs.setdefault(id_vers_nom(c), set()).add(p)

# --- 3. MoocRadar : nom de concept -> problèmes ---
nom_vers_pbs_r = {}
for p, cs in concepts_par_pb.items():
    for c in cs:
        nom_vers_pbs_r.setdefault(c.strip(), set()).add(p)

def resume(sous_ensemble, etiquette):
    concepts = {x for _, a, b, _ in sous_ensemble for x in (a, b)}
    c_x = {c for c in concepts if c in concept_pbs}
    c_r = {c for c in concepts if c in nom_vers_pbs_r}
    pbs_r = set().union(*[nom_vers_pbs_r[c] for c in c_r]) if c_r else set()
    return {
        'sous_ensemble': etiquette,
        'paires': len(sous_ensemble),
        'concepts': len(concepts),
        'X_concepts_avec_exercices': len(c_x),
        'X_paires_deux_cotes_avec_exercices': sum(1 for _, a, b, _ in sous_ensemble if a in concept_pbs and b in concept_pbs),
        'X_problemes_concernes': len(set().union(*[concept_pbs[c] for c in c_x])) if c_x else 0,
        'R_concepts_trouves_dans_MoocRadar': len(c_r),
        'R_paires_deux_cotes_dans_MoocRadar': sum(1 for _, a, b, _ in sous_ensemble if a in nom_vers_pbs_r and b in nom_vers_pbs_r),
        'R_problemes_concernes': len(pbs_r),
        'R_interactions_concernees': sum(inter_par_pb[p] for p in pbs_r),
    }

lignes = [resume(paires, 'TOUT')]
for dom in DOMAINES:
    lignes.append(resume([p for p in paires if p[0] == dom], f'domaine={dom}'))
for k, c in drapeaux.items():
    for val in c:
        lignes.append(resume([p for p in paires if p[3].get(k) == val], f'{k}={val}'))

tableau = pd.DataFrame(lignes).set_index('sous_ensemble')
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 50)
print(tableau.T)
tableau.to_csv(os.path.join(RESULTATS, 'test_decisif.csv'))


**Cellule 6 (optionnelle, longue)** — compte les réponses d'apprenants de MOOCCubeX sur les problèmes reliés à des concepts ayant des prérequis.
Elle lit les 21 Go en flux depuis le zip : comptez 30 à 90 minutes. Lancez-la seulement si la cellule 5 montre des paires côté MOOCCubeX (colonnes `X_...`).

In [ ]:
# Cellule 6 — Réponses MOOCCubeX sur les problèmes concernés (lecture en flux)
import time
concepts_prereq = {x for _, a, b, _ in paires for x in (a, b)}
pbs_cibles = set().union(*[concept_pbs[c] for c in concepts_prereq if c in concept_pbs]) if concepts_prereq else set()
print('Problèmes ciblés :', len(pbs_cibles))
cibles_b = {p.encode() for p in pbs_cibles}
CLE = b'"problem_id": "'

compte_pb, apprenants, total, t0 = Counter(), set(), 0, time.time()
with zipfile.ZipFile(ZIP_X) as z:
    nom_up = next(n for n in z.namelist() if n.endswith('user-problem.json'))
    with z.open(nom_up) as f:
        for l in f:
            total += 1
            i = l.find(CLE)
            if i < 0:
                continue
            j = l.find(b'"', i + len(CLE))
            pid = l[i + len(CLE):j]
            if pid in cibles_b:
                compte_pb[pid] += 1
                k = l.find(b'"user_id": "')
                if k >= 0:
                    apprenants.add(l[k + 12:l.find(b'"', k + 12)])
            if total % 20_000_000 == 0:
                print(f'{total/1e6:.0f} M lignes lues, {sum(compte_pb.values())/1e6:.2f} M retenues, {(time.time()-t0)/60:.1f} min')

stats_x = {
    'lignes_user_problem': total,
    'reponses_sur_problemes_cibles': sum(compte_pb.values()),
    'problemes_cibles_avec_reponses': len(compte_pb),
    'apprenants_concernes': len(apprenants),
}
print(stats_x)

In [ ]:
# Cellule 7 — Résumé à copier dans la conversation
resultat = {
    'moocradar': stats_r,
    'champs_drapeaux_prerequis': {k: {str(a): b for a, b in c.items()} for k, c in drapeaux.items()},
    'test_decisif': tableau.reset_index().to_dict(orient='records'),
    'mooccubex_reponses': globals().get('stats_x', 'cellule 6 non exécutée'),
}
with open(os.path.join(RESULTATS, 'test_decisif.json'), 'w', encoding='utf-8') as f:
    json.dump(resultat, f, ensure_ascii=False, indent=2, default=str)
print(json.dumps(resultat, ensure_ascii=False, indent=2, default=str))